In [ ]:
import os

folder_path = "/content/drive/MyDrive/Upwork/patrick-bwn-project/MindBodyOnline"
json_files = []

# Check if the folder exists
if os.path.exists(folder_path):
    # Iterate over all files and directories in the specified path
    for filename in os.listdir(folder_path):
        # Check if the file ends with '.json'
        if filename.endswith('.json'):
            json_files.append(filename)

    if json_files:
        print(f"JSON files found in '{folder_path}':")
        for json_file in json_files:
            print(f"- {json_file}")
    else:
        print(f"No JSON files found in '{folder_path}'.")
else:
    print(f"The folder '{folder_path}' does not exist. Please make sure the path is correct and Google Drive is mounted.")

JSON files found in '/content/drive/MyDrive/Upwork/patrick-bwn-project/MindBodyOnline':
- 20260706_145740_145_combined_result.json
- 20260707_031801_299_combined_result.json
- 300_cities_locations.json
- 301-600_cities_locations.json
- 20260707_034634_325_combined_result.json
- 601-1000_cities_locations.json
- 20260708_112301_377_combined_result.json
- 150_cities_locations.json
- 1001-1952_cities_locations.json
- 20260711_152757_952_combined_result.json


In [ ]:
import json

combined_results = []

# Filter for files containing '_combined_result'
combined_json_filenames = [f for f in json_files if '_combined_result' in f]

if combined_json_filenames:
    print(f"Combining the following files: {combined_json_filenames}")
    for filename in combined_json_filenames:
        file_path = os.path.join(folder_path, filename)
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
                if isinstance(data, list):
                    combined_results.extend(data)
                else:
                    combined_results.append(data) # Handle single JSON objects by adding them to the list
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from {filename}: {e}")
        except Exception as e:
            print(f"An unexpected error occurred while reading {filename}: {e}")

    output_file_path = os.path.join('/content', 'combined_results.json')
    with open(output_file_path, 'w') as outfile:
        json.dump(combined_results, outfile, indent=4)
    print(f"Successfully combined {len(combined_json_filenames)} JSON files into '{output_file_path}'.")
    print(f"Total items in the combined JSON: {len(combined_results)}")
else:
    print("No JSON files containing '_combined_result' were found to combine.")

Combining the following files: ['20260706_145740_145_combined_result.json', '20260707_031801_299_combined_result.json', '20260707_034634_325_combined_result.json', '20260708_112301_377_combined_result.json', '20260711_152757_952_combined_result.json']
Successfully combined 5 JSON files into '/content/combined_results.json'.
Total items in the combined JSON: 2384


In [ ]:
import json

# Define the path to the combined results file
combined_file_path = '/content/combined_results.json'

# Check if the file exists
if os.path.exists(combined_file_path):
    try:
        with open(combined_file_path, 'r') as f:
            all_combined_data = json.load(f)

        # Check if there are any records and display the first one
        if all_combined_data:
            print("First record from combined_results.json:")
            display(all_combined_data[0])
        else:
            print("The combined_results.json file is empty.")

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from {combined_file_path}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while reading {combined_file_path}: {e}")
else:
    print(f"The file '{combined_file_path}' does not exist.")

First record from combined_results.json:


{'city': 'Albuquerque',
 'hierarchy': 'New Mexico, United States',
 'latitude': 35.088856,
 'longitude': -106.6491,
 'result': {'data': [{'id': '644891',
    'type': 'location',
    'attributes': {'categoryTypes': None,
     'categories': None,
     'subcategories': None,
     'distance': 0.3703305778754502,
     'slug': 'cereset-albuquerque',
     'logo': 'https://prodapilymberfitness.mindbodyonline.com/organizations/103921/original_logo.gif?1763628683',
     'address': '500 Marquette Avenue Northwest Ste 1244',
     'address2': '',
     'city': 'Albuquerque',
     'state': 'NM',
     'county': 'Bernalillo County',
     'country': 'United States',
     'postalCode': '87102',
     'phone': '15059999277',
     'timezone': 'America/Denver',
     'locale': 'en_US',
     'latLon': '35.0880857,-106.6530597',
     'neighborhood': 'Downtown',
     'amenities': None,
     'dynamicPriceable': 0,
     'currencyCode': 'USD',
     'inventoryRefJson': '{"mb_site_id":5746080,"mb_location_id":1,"mb_m

In [1]:
"""
flatten_studios.py
===================
Flattens the nested `combined_results.json` (city -> result.data[] -> attributes)
into a single tidy table of studio/gym/wellness locations.

Designed to run in Google Colab. Just upload `combined_results.json` and run.

Usage in Colab:
    from google.colab import files
    uploaded = files.upload()   # upload combined_results.json
    !python flatten_studios.py combined_results.json

Or simply paste this whole script into a cell and run — it will look for
'combined_results.json' in the current directory, or prompt you to upload it.
"""

import json
import sys
import os
import traceback
from datetime import datetime

try:
    import pandas as pd
except ImportError:
    os.system("pip install pandas --quiet")
    import pandas as pd


# --------------------------------------------------------------------------- #
#  Pretty logging helpers
# --------------------------------------------------------------------------- #
class Log:
    OK = "✅"
    WARN = "⚠️ "
    ERR = "❌"
    INFO = "ℹ️ "
    STEP = "▶️ "

    @staticmethod
    def line(char="─", n=70):
        print(char * n)

    @staticmethod
    def header(title):
        Log.line("═")
        print(f"  {title}")
        Log.line("═")

    @staticmethod
    def step(msg):
        print(f"{Log.STEP} {msg}")

    @staticmethod
    def ok(msg):
        print(f"{Log.OK} {msg}")

    @staticmethod
    def warn(msg):
        print(f"{Log.WARN} {msg}")

    @staticmethod
    def err(msg):
        print(f"{Log.ERR} {msg}")

    @staticmethod
    def info(msg):
        print(f"{Log.INFO} {msg}")


# --------------------------------------------------------------------------- #
#  Helpers to safely pull values out of messy / inconsistent records
# --------------------------------------------------------------------------- #
def safe_get(d, key, default=None):
    """Get a key from a dict, tolerating None dicts."""
    if not isinstance(d, dict):
        return default
    val = d.get(key, default)
    return val if val is not None else default


def join_list(val, sep=", "):
    """Turn a list of strings (or None) into a joined string."""
    if not val:
        return ""
    if isinstance(val, list):
        return sep.join(str(v) for v in val if v is not None)
    return str(val)


def join_named_list(val, sep=", "):
    """Turn a list of dicts like [{'name': 'Parking'}, ...] into a joined string."""
    if not val or not isinstance(val, list):
        return ""
    names = []
    for item in val:
        if isinstance(item, dict) and "name" in item:
            names.append(str(item["name"]))
        elif isinstance(item, str):
            names.append(item)
    return sep.join(names)


def parse_lat_lon(lat_lon_str):
    """'35.088,-106.653' -> (35.088, -106.653)"""
    if not lat_lon_str or not isinstance(lat_lon_str, str):
        return (None, None)
    try:
        parts = lat_lon_str.split(",")
        return (float(parts[0].strip()), float(parts[1].strip()))
    except Exception:
        return (None, None)


def parse_inventory_ref(ref_json_str):
    """inventoryRefJson is a JSON-encoded string; extract useful sub-fields."""
    if not ref_json_str:
        return {}
    try:
        return json.loads(ref_json_str)
    except Exception:
        return {}


def first_gallery_image(gallery_images):
    if isinstance(gallery_images, list) and len(gallery_images) > 0:
        first = gallery_images[0]
        if isinstance(first, dict):
            return first.get("url", "")
    return ""


# --------------------------------------------------------------------------- #
#  Core flattening logic
# --------------------------------------------------------------------------- #
def flatten_location(city_record, location_record):
    """Build one flat dict for a single studio/location entry."""
    attrs = safe_get(location_record, "attributes", {}) or {}

    lat, lon = parse_lat_lon(attrs.get("latLon"))
    inv_ref = parse_inventory_ref(attrs.get("inventoryRefJson"))

    flat = {
        # --- search context (which city query returned this result) ---
        "search_city": city_record.get("city"),
        "search_hierarchy": city_record.get("hierarchy"),
        "search_latitude": city_record.get("latitude"),
        "search_longitude": city_record.get("longitude"),

        # --- identifiers ---
        "id": location_record.get("id"),
        "type": location_record.get("type"),
        "slug": attrs.get("slug"),

        # --- naming ---
        "name": attrs.get("name"),
        "location_name": attrs.get("locationName"),
        "description": attrs.get("description"),
        "location_description": attrs.get("locationDescription"),

        # --- address ---
        "address": attrs.get("address"),
        "address2": attrs.get("address2"),
        "city": attrs.get("city"),
        "state": attrs.get("state"),
        "county": attrs.get("county"),
        "country": attrs.get("country"),
        "postal_code": attrs.get("postalCode"),
        "neighborhood": attrs.get("neighborhood"),

        # --- geo ---
        "latitude": lat,
        "longitude": lon,
        "distance_miles": attrs.get("distance"),

        # --- contact ---
        "phone": attrs.get("phone"),
        "timezone": attrs.get("timezone"),
        "locale": attrs.get("locale"),

        # --- categorization ---
        "category_types": join_list(attrs.get("categoryTypes")),
        "categories": join_list(attrs.get("categories")),
        "subcategories": join_list(attrs.get("subcategories")),
        "secondary_categories": join_list(attrs.get("secondaryCategories")),

        # --- ratings / commerce ---
        "total_deals": attrs.get("totalDeals"),
        "total_ratings": attrs.get("totalRatings"),
        "average_rating": attrs.get("averageRating"),
        "dynamic_priceable": attrs.get("dynamicPriceable"),
        "online_store_active": attrs.get("onlineStoreActive"),
        "cm_membership_bookable": attrs.get("cmMembershipBookable"),
        "allow_mobile_checkin": attrs.get("allowMobileCheckin"),
        "currency_code": attrs.get("currencyCode"),
        "currency_symbol": attrs.get("currencySymbol"),

        # --- amenities / identity ---
        "amenities": join_list(attrs.get("amenities")),
        "categorized_amenities": join_named_list(attrs.get("categorizedAmenities")),
        "business_identifies_as": join_named_list(attrs.get("businessIdentifiesAs")),
        "crowd_safe_as": join_named_list(attrs.get("crowdSafeAs")),

        # --- media ---
        "logo": attrs.get("logo"),
        "site_logo": attrs.get("siteLogo"),
        "mobile_logo": attrs.get("mobileLogo"),
        "stock_image": attrs.get("stockImage"),
        "gallery_image_count": len(attrs.get("galleryImages") or []),
        "first_gallery_image": first_gallery_image(attrs.get("galleryImages")),

        # --- inventory / source system ---
        "inventory_source": attrs.get("inventorySource"),
        "mb_site_id": inv_ref.get("mb_site_id"),
        "mb_location_id": inv_ref.get("mb_location_id"),
        "mb_master_location_id": inv_ref.get("mb_master_location_id"),
        "booker_location_id": inv_ref.get("booker_location_id"),

        "country_code": attrs.get("countryCode"),
    }
    return flat


# --------------------------------------------------------------------------- #
#  Main driver
# --------------------------------------------------------------------------- #
def flatten_file(input_path, output_csv="studios_flattened.csv",
                  output_json="studios_flattened.json"):
    Log.header("STUDIO / GYM JSON FLATTENER")
    start_time = datetime.now()

    # 1. Load the file -------------------------------------------------------
    Log.step(f"Loading input file: {input_path}")
    if not os.path.exists(input_path):
        Log.err(f"File not found: {input_path}")
        raise FileNotFoundError(input_path)

    try:
        with open(input_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        Log.ok(f"Loaded JSON successfully ({os.path.getsize(input_path)/1024:.1f} KB)")
    except json.JSONDecodeError as e:
        Log.err(f"Invalid JSON in {input_path}: {e}")
        raise
    except Exception as e:
        Log.err(f"Unexpected error reading file: {e}")
        raise

    # Some exports wrap everything in a list, others might be a single dict
    if isinstance(raw_data, dict):
        city_records = [raw_data]
    elif isinstance(raw_data, list):
        city_records = raw_data
    else:
        Log.err(f"Unexpected top-level JSON type: {type(raw_data)}")
        raise TypeError("Top-level JSON must be a list or a dict")

    Log.info(f"Found {len(city_records)} city/search record(s) to process")
    Log.line()

    # 2. Flatten every location within every city record ---------------------
    all_rows = []
    total_locations = 0
    skipped_cities = 0
    skipped_locations = 0
    error_log = []

    for city_idx, city_record in enumerate(city_records, start=1):
        try:
            city_name = city_record.get("city", f"<unknown #{city_idx}>")
            result = safe_get(city_record, "result", {}) or {}
            data_list = safe_get(result, "data", []) or []
            meta = safe_get(result, "meta", {}) or {}
            found_count = meta.get("found", len(data_list))

            Log.step(
                f"[{city_idx}/{len(city_records)}] {city_name}: "
                f"{len(data_list)} location(s) in payload (meta.found={found_count})"
            )

            if not data_list:
                Log.warn(f"  No locations found for '{city_name}' — skipping")
                skipped_cities += 1
                continue

            city_success = 0
            for loc_idx, location_record in enumerate(data_list, start=1):
                try:
                    flat_row = flatten_location(city_record, location_record)
                    all_rows.append(flat_row)
                    city_success += 1
                    total_locations += 1
                except Exception as e:
                    skipped_locations += 1
                    err_msg = (f"City '{city_name}' location #{loc_idx} "
                               f"(id={location_record.get('id', '?')}): {e}")
                    error_log.append(err_msg)
                    Log.warn(f"  Skipped a location due to error: {e}")

            Log.ok(f"  Flattened {city_success}/{len(data_list)} locations for '{city_name}'")

        except Exception as e:
            skipped_cities += 1
            error_log.append(f"City record #{city_idx}: {e}")
            Log.err(f"Failed to process city record #{city_idx}: {e}")
            Log.err(traceback.format_exc(limit=1))
            continue

    Log.line()

    # 3. Build DataFrame -------------------------------------------------------
    if not all_rows:
        Log.err("No rows were flattened — nothing to save. Aborting.")
        raise ValueError("No data extracted from input JSON")

    Log.step("Building DataFrame")
    df = pd.DataFrame(all_rows)

    # De-duplicate on 'id' just in case the same location appears in multiple
    # city searches (common with radius-based location searches)
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)
    after_dedup = len(df)
    if before_dedup != after_dedup:
        Log.info(f"Removed {before_dedup - after_dedup} duplicate location(s) (same id)")

    Log.ok(f"DataFrame built: {df.shape[0]} rows x {df.shape[1]} columns")

    # 4. Save outputs ------------------------------------------------------
    Log.step(f"Saving CSV to: {output_csv}")
    try:
        df.to_csv(output_csv, index=False)
        Log.ok(f"CSV saved ({os.path.getsize(output_csv)/1024:.1f} KB)")
    except Exception as e:
        Log.err(f"Failed to save CSV: {e}")

    Log.step(f"Saving JSON to: {output_json}")
    try:
        df.to_json(output_json, orient="records", indent=2)
        Log.ok(f"JSON saved ({os.path.getsize(output_json)/1024:.1f} KB)")
    except Exception as e:
        Log.err(f"Failed to save JSON: {e}")

    # 5. Summary -------------------------------------------------------------
    elapsed = (datetime.now() - start_time).total_seconds()
    Log.header("SUMMARY")
    print(f"  Cities processed        : {len(city_records)}")
    print(f"  Cities skipped          : {skipped_cities}")
    print(f"  Locations flattened     : {total_locations}")
    print(f"  Locations skipped       : {skipped_locations}")
    print(f"  Unique rows in output   : {len(df)}")
    print(f"  Columns in output       : {len(df.columns)}")
    print(f"  Time elapsed            : {elapsed:.2f}s")

    if error_log:
        Log.line()
        Log.warn(f"{len(error_log)} error(s) occurred during processing:")
        for e in error_log[:20]:
            print(f"    - {e}")
        if len(error_log) > 20:
            print(f"    ... and {len(error_log) - 20} more")

    Log.line("═")
    Log.ok("Done! 🎉")
    Log.line("═")

    return df


# --------------------------------------------------------------------------- #
#  Entry point (Colab / CLI friendly)
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    """
    # Allow: python flatten_studios.py path/to/file.json
    input_file = sys.argv[1] if len(sys.argv) > 1 else "/content/combined_results.json"

    if not os.path.exists(input_file):
        Log.warn(f"'{input_file}' not found in current directory.")
        try:
            # If running inside Colab, offer an upload widget
            from google.colab import files
            Log.info("Running in Google Colab — please upload your JSON file now.")
            uploaded = files.upload()
            input_file = list(uploaded.keys())[0]
        except ImportError:
            Log.err("Not running in Colab and file not found. "
                     "Pass the path as an argument: python flatten_studios.py your_file.json")
            sys.exit(1)
    """
    df = flatten_file(all_combined_data)

    # Quick peek at the result
    Log.info("Preview of first 5 rows:")
    try:
        from IPython.display import display
        display(df.head())
    except Exception:
        print(df.head().to_string())

NameError: name 'all_combined_data' is not defined